# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Convert metadata to dict for printing
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\nDescription: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
print("Record Sets (by @id):")
for rs in dataset.record_sets:
    print(f"  - ID: {rs['@id']}, Name: {rs.get('name', '(no name)')}")

# Show fields and columns for each record set
for rs in dataset.record_sets:
    print(f"\nFields for record set {rs['@id']}:")
    for field in rs.get('field', []):
        # field can be a dict or @id string depending on expansion
        if isinstance(field, dict):
            field_id = field.get('@id', str(field))
            name = field.get('name', '')
        else:
            field_id = field
            name = ''
        print(f"    - Field ID: {field_id} {f'Name: {name}' if name else ''}")
    # Try to print columns if any
    if rs.get('column', []):
        print(f"Columns for record set {rs['@id']}:")
        for col in rs.get('column', []):
            if isinstance(col, dict):
                col_id = col.get('@id', str(col))
                name = col.get('name', '')
            else:
                col_id = col
                name = ''
            print(f"    - Column ID: {col_id} {f'Name: {name}' if name else ''}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List record set IDs
record_sets = [rs['@id'] for rs in dataset.record_sets]
print(f"Found record sets: {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    try:
        # Load all records
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id}. Shape: {dataframes[record_set_id].shape}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# Show columns for first record set
if record_sets:
    main_rs = record_sets[0]
    if main_rs in dataframes:
        print(f"Columns in record set {main_rs}:\n{dataframes[main_rs].columns.tolist()}")
        display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For the main record set, pick a numeric field
main_record_set_id = record_sets[0] if record_sets else None

# List columns to help pick a numeric field
print(f"Columns in main record set:", dataframes[main_record_set_id].columns.tolist())

# Let's try common protein quant columns: MW (Molecular Weight), or PeptidesFound, or Coverage, etc.
# We'll try to find a good candidate
numeric_fields = []
df = dataframes[main_record_set_id]
# Find numeric dtype columns
for col in df.columns:
    # Try convert to numeric to test
    try:
        pd.to_numeric(df[col], errors='raise')
        numeric_fields.append(col)
    except:
        continue

print(f"Numeric candidate fields found: {numeric_fields}")

if numeric_fields:
    # Use the first numeric field as example
    numeric_field = numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")

    # Threshold-based filtering (e.g. proteins with value > threshold)
    threshold = df[numeric_field].astype(float).mean()  # Just for demo, mean as threshold
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()

    print(f"Filtered records with {numeric_field} > {threshold:.2f}")
    display(filtered_df.head())

    # Normalization (z-score)
    filtered_df[numeric_field + '_normalized'] = (
        pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Group by a likely categorical field e.g. 'SampleType' if present
    group_candidates = [col for col in df.columns if df[col].nunique() < 10 and col != numeric_field]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print("Grouped data (mean):")
        display(grouped_df)
    else:
        print("No suitable grouping field found.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    # Histogram of the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping field exists, boxplot
    if group_candidates:
        plt.figure(figsize=(9, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we have:

- Loaded the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using `mlcroissant`.
- Explored the dataset structure, available record sets, and fields using their `@id` references.
- Extracted records from the main record set into a DataFrame.
- Performed basic exploratory analysis on numeric fields (e.g., filtering by threshold, normalization, grouping).
- Visualized data distributions and relationships.

You are encouraged to further explore the provided fields and try advanced visualizations or modeling tasks tailored to your research questions.